# Práctica guiada — Inferencia y contratos

Este notebook es para el alumnado. Completa las celdas marcadas con `TODO` y responde las preguntas antes de mirar la solución docente.

**Objetivo:** pasar de una fila CSV a una especificación clara para el módulo de inferencia de la clase 2.

## Cómo trabajar

1. Copia esta carpeta como tu espacio de trabajo.
2. Ejecuta `uv sync` desde `problem/starter/` cuando empiece la clase 2.
3. En esta práctica no implementes todavía el CLI completo.
4. Trabaja en parejas y cambia quién escribe cada diez minutos.
5. No abras `solutions/` hasta el debrief.

In [1]:
from dataclasses import dataclass
import csv
from pathlib import Path
from typing import Literal

from pydantic import BaseModel, ConfigDict, Field, ValidationError

DATA_PATH = Path("../../../../../../assets/03-wine-quality/inference_samples.csv")
print(DATA_PATH)
print(DATA_PATH.exists())

..\..\..\..\..\..\assets\03-wine-quality\inference_samples.csv
True


## 1. Leer los datos nuevos

### Práctica A — Inspección del CSV

Completa la celda siguiente para:

- Leer el CSV con `csv.DictReader`.
- Imprimir el número de filas.
- Imprimir las columnas.
- Guardar la primera fila en una variable llamada `row`.

In [2]:
# TODO: implementa la lectura del CSV.
# Pista: usa DATA_PATH.open(="newline", encoding="utf-8")

with DATA_PATH.open(newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    rows = list(reader)

    print("Número de filas:", len(rows))
    print("Columnas:", reader.fieldnames)

row = rows[0]
print("Primera fila:", row)

Número de filas: 5
Columnas: ['sample_id', 'fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'ph', 'sulphates', 'alcohol']
Primera fila: {'sample_id': 'red-001', 'fixed_acidity': '7.4', 'volatile_acidity': '0.7', 'citric_acid': '0.0', 'residual_sugar': '1.9', 'chlorides': '0.076', 'free_sulfur_dioxide': '11.0', 'total_sulfur_dioxide': '34.0', 'density': '0.9978', 'ph': '3.51', 'sulphates': '0.56', 'alcohol': '9.4'}


### Preguntas

1. ¿Qué campo identifica la muestra pero no es una característica del modelo? 
sample_id. Sirve para saber que muestra estamos procesando no entra en el modelo.
2. ¿Cuántas características numéricas hay?
hay 11 características numéricas 
3. ¿Qué columna extra añadirías para probar el contrato?
cualquier columna para que el contrato la rechaze
4. ¿Qué valor de `ph` sería claramente inválido?
uno que salga del rango 2.5-4.5

## 2. Construir el contrato con Pydantic

El contrato del modelo contiene estas características. Todas son de tipo `float` y obligatorias (no admiten valores ausentes); los límites se incluyen:

| Campo | Unidad | Rango orientativo |
|---|---|---|
| `fixed_acidity` | g/dm³ de ácido tartárico | 0–20 |
| `volatile_acidity` | g/dm³ de ácido acético | 0–2 |
| `citric_acid` | g/dm³ | 0–2 |
| `residual_sugar` | g/dm³ | 0–20 |
| `chlorides` | g/dm³ de cloruro de sodio | 0–1 |
| `free_sulfur_dioxide` | mg/dm³ | 0–100 |
| `total_sulfur_dioxide` | mg/dm³ | 0–300 |
| `density` | g/cm³ | 0.98–1.01 |
| `ph` | adimensional | 2.5–4.5 |
| `sulphates` | g/dm³ de sulfato de potasio | 0–3 |
| `alcohol` | % vol. | 5–20 |


In [3]:
class WineQualityRequestPydantic(BaseModel):
    model_config = ConfigDict(extra="forbid")


    fixed_acidity: float = Field(ge=0, le=20)
    volatile_acidity: float = Field(ge=0, le=2)
    citric_acid: float = Field(ge=0, le=2)
    residual_sugar: float = Field(ge=0, le=20)
    chlorides: float = Field(ge=0, le=1)
    free_sulfur_dioxide: float = Field(ge=0, le=100)
    total_sulfur_dioxide: float = Field(ge=0, le=300)
    density: float = Field(ge=0.98, le=1.01)
    ph: float = Field(ge=2.5, le=4.5)
    sulphates: float = Field(ge=0, le=3)
    alcohol: float = Field(ge=5, le=20)

# TODO: separa sample_id de la fila y valida solo las características del modelo.
model_row = row.copy()
sample_id = model_row.pop("sample_id")

request = WineQualityRequestPydantic.model_validate(model_row)

### Práctica B — Validación

Completa una prueba para cada caso:

- Una fila válida.
- Una columna `unexpected_field`.
- Un `ph` fuera de rango.

Antes de ejecutar, escribe qué error esperas.

In [4]:
# TODO: completa los casos y captura ValidationError.

# Caso 1: fila válida
valid_request = WineQualityRequestPydantic.model_validate(model_row)
print(valid_request)

# Caso 2: campo extra
extra_row = model_row.copy()
extra_row["unexpected_field"] = 1

try:
    WineQualityRequestPydantic.model_validate(extra_row)
except ValidationError as error:
    print(error)
    
# Caso 3: ph fuera de rango
invalid_ph_row = model_row.copy()
invalid_ph_row["ph"] = 9.5

try:
    WineQualityRequestPydantic.model_validate(invalid_ph_row)
except ValidationError as error:
    print(error)

fixed_acidity=7.4 volatile_acidity=0.7 citric_acid=0.0 residual_sugar=1.9 chlorides=0.076 free_sulfur_dioxide=11.0 total_sulfur_dioxide=34.0 density=0.9978 ph=3.51 sulphates=0.56 alcohol=9.4
1 validation error for WineQualityRequestPydantic
unexpected_field
  Extra inputs are not permitted [type=extra_forbidden, input_value=1, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden
1 validation error for WineQualityRequestPydantic
ph
  Input should be less than or equal to 4.5 [type=less_than_equal, input_value=9.5, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


## 3. Comparar con `dataclass`

Construye la misma estructura con `dataclass`. No añadas todavía validación manual: observa qué garantiza Python y qué no garantiza.

In [5]:
@dataclass(frozen=True, slots=True)
class WineQualityRequestDataclass:
    # TODO: declara las mismas 11 características.
    fixed_acidity: float  
    volatile_acidity: float 
    citric_acid: float 
    residual_sugar: float 
    chlorides: float 
    free_sulfur_dioxide: float
    total_sulfur_dioxide: float
    density: float
    ph: float 
    sulphates: float
    alcohol: float 
# TODO: crea una instancia válida y prueba qué ocurre con ph=99.0.

valid_dataclass = WineQualityRequestDataclass(**model_row)
print(valid_dataclass)

invalid_row = model_row.copy()
invalid_row["ph"] = 9.5

invalid_dataclass = WineQualityRequestDataclass(**invalid_row)
print(invalid_dataclass)

WineQualityRequestDataclass(fixed_acidity='7.4', volatile_acidity='0.7', citric_acid='0.0', residual_sugar='1.9', chlorides='0.076', free_sulfur_dioxide='11.0', total_sulfur_dioxide='34.0', density='0.9978', ph='3.51', sulphates='0.56', alcohol='9.4')
WineQualityRequestDataclass(fixed_acidity='7.4', volatile_acidity='0.7', citric_acid='0.0', residual_sugar='1.9', chlorides='0.076', free_sulfur_dioxide='11.0', total_sulfur_dioxide='34.0', density='0.9978', ph=9.5, sulphates='0.56', alcohol='9.4')


### Decisión de diseño

Completa la frase:

> Usaría Pydantic en casos donde no tenga las validados los tipos porque así puedo validarlos. Usaría `dataclass` en caso de estar ya validados porque es más comodo y profesional.

## 4. Definir el orden del vector

El modelo no recibe un diccionario. Recibe una lista numérica con un orden estable.

In [6]:
# Orden exacto de las características que consume el modelo.
FEATURE_NAMES = (
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol",
)

# TODO: construye vector usando getattr(request, name).
vector =  [getattr(request, name) for name in FEATURE_NAMES] 

### Práctica C — Error de orden

Intercambia mentalmente `density` y `alcohol`. Responde:

- ¿El vector sigue teniendo 11 valores? Correcto
- ¿El clasificador detectaría necesariamente el error? No, ya que no sabe interpretar el cambio, va a pensar que el valor corresponde a la columna asignada para esa posición.
- ¿Cómo evitarías que cada script definiera un orden distinto? 
establecería FEATURE_NAMES en un orden específico en el que luego tienes que pasar los valores en orden para que funcione.

## 5. Contrato de salida

Diseña la estructura que devolverá el módulo. Debe permitir responder:

- ¿Qué muestra se ha procesado?
- ¿Qué categoría se ha predicho?
- ¿Con qué confianza?
- ¿Qué versión de modelo y preprocesado se utilizaron?

In [ ]:
class WineQualityPredictionPydantic(BaseModel):
    # TODO: declara los campos de salida y sus restricciones.
    sample_id: str
    quality_band: Literal["needs_review", "acceptable", "excellent"]
    confidence: float = Field(ge=0, le=1)
    model_version: str
    preprocessing_version: str


# TODO: escribe un ejemplo de salida para red-001.
prediction = WineQualityPredictionPydantic(
    sample_id="red-001",
    quality_band="acceptable",
    confidence=0.85,
    model_version="1.0",
    preprocessing_version="1.0",
)

print(prediction)

## 6. Demo del docente: observar la inferencia

El docente ejecutará el notebook resuelto y el comando de referencia. Mientras tanto, anota el recorrido:

```text
CSV → contrato → preprocesado → modelo cargado → predicción
```

No copies la implementación. Tu código de producción empieza en la clase 2 con el starter.

## Entrega

Entrega una captura o exportación con:

- La tabla de contrato.
- Las respuestas de las prácticas A, B y C.
- El orden de las 11 características.
- Un ejemplo de entrada inválida.
- Un contrato de salida.
- Tu plan de implementación para `contracts.py`, `preprocess.py`, `inference.py` y `predict_file.py`.

Plan de implementación para la clase 2:

- contracts.py: definir los contratos Pydantic de entrada y salida.

- preprocess.py: transformar una petición validada en el vector de 11 características respetando FEATURE_NAMES.

- inference.py: cargar el modelo entrenado y realizar la predicción.

- predict_file.py: leer el CSV, validar cada fila, ejecutar la inferencia y generar el fichero de salida.

Tres reglas explícitas de aceptación para el módulo local:

1. Con el CSV válido de cinco muestras, el CLI termina correctamente y genera cinco predicciones con `sample_id`, `quality_band` (solo `needs_review`, `acceptable` o `excellent`), `confidence` entre 0 y 1 y las versiones de modelo y preprocesado. El vector contiene las 11 características en el orden de `FEATURE_NAMES`, sin `sample_id`.
2. Si una fila incluye `unexpected_field`, la validación de entrada la rechaza antes de inferir y el CLI termina con error sin crear el CSV de salida.
3. Si la segunda fila tiene `ph=9.5`, la validación de entrada la rechaza y el CLI termina con error sin crear un CSV parcial, aunque la primera fila sea válida.
